# CEFR Multi-Prefix Tuning — 68M Parameter-Matched Variant

This notebook trains and evaluates the **~68M CEFR Multi-Prefix Tuning model** used in the thesis.

The experiment is designed as a parameter-matched comparison with the Standard Prefix-Tuning baseline. Both methods use the frozen `meta-llama/Llama-3.1-8B-Instruct` backbone and approximately 68M trainable parameters.

The CEFR Multi-Prefix controller introduces six CEFR-specific prefix banks, one for each proficiency level (A1–C2). Each level uses 30 virtual prefix tokens. The controller maps CEFR-specific prefix embeddings into the full per-layer key/value prefix representation required by Llama-3.1-8B.

To closely match the parameter count of Standard Prefix-Tuning, the CEFR prefix token dimension is reduced to 1024, corresponding to the 8 key/value heads × 128 head dimension of Llama-3.1-8B.

The notebook includes:

- construction of the parameter-matched CEFR Multi-Prefix controller;
- training on the Balanced CEFR Steering Subset;
- validation-loss and perplexity tracking;
- hardware profiling and parameter-count verification;
- publication of the trained controller to Hugging Face;
- generation on the final In-Domain Evaluation Prompt Matrix;
- evaluation with the primary Joint-Loss CEFR evaluator.

The final model contains **68,408,320 trainable parameters**, only **153,600 (+0.225%)** more than the Standard Prefix-Tuning reference with 68,254,720 parameters.

In [ ]:
# =========================================================
# 0. ENVIRONMENT SETUP & DEPENDENCIES
# =========================================================

!pip install -q transformers peft torch datasets huggingface_hub accelerate scikit-learn textstat spacy hf_transfer nvidia-ml-py "torchao>=0.16.0"
!python -m spacy download en_core_web_sm

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import gc
import json
import math
import time
import threading
import pynvml
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_cosine_schedule_with_warmup,
    set_seed
)

from transformers.cache_utils import DynamicCache

from huggingface_hub import login, HfApi
from google.colab import drive, userdata
from tqdm.auto import tqdm


# =========================================================
# 1. REPRODUCIBILITY
# =========================================================

SEED = 42

set_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# =========================================================
# 2. MOUNT GOOGLE DRIVE
# =========================================================

drive.mount("/content/drive")


# =========================================================
# 3. EXPERIMENT PATHS
# =========================================================
#
# These paths are intentionally different from:
#
#   CEFR Prefix-Tuning ~286M
#   CEFR Prefix-Tuning ~537M
#   Standard Prefix-Tuning ~68M
#
# so nothing is overwritten.
#
# =========================================================

BALANCED_CSV_PATH = (
    "/content/drive/MyDrive/Your_Path/"
    "balanced_cefr_steering_subset.csv"
)

HF_REPO_ID = (
    "MohammadKhosravi/"
    "llama3.1-8b-cefr-prefix-tuning-68m-param-matched"
)

DRIVE_LOG_DIR = (
    "/content/drive/MyDrive/Your_Path/"
    "cefr_prefix_tuning_68m_training_logs"
)

LOCAL_OUTPUT_DIR = (
    "./cefr_prefix_tuning_68m_param_matched_export"
)

os.makedirs(
    DRIVE_LOG_DIR,
    exist_ok=True
)

os.makedirs(
    LOCAL_OUTPUT_DIR,
    exist_ok=True
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 103.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 125.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Mounted at /content/drive


In [ ]:
# =========================================================
# 4. HUGGING FACE LOGIN
# =========================================================

try:

    hf_token = userdata.get("HF_TOKEN")

    login(
        token=hf_token
    )

except Exception:
    pass


# =========================================================
# 5. DEVICE
# =========================================================

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    f"Using execution device: {device}"
)

if torch.cuda.is_available():

    print(
        f"Device Name: "
        f"{torch.cuda.get_device_name(0)}"
    )

Using execution device: cuda
Device Name: NVIDIA A100-SXM4-80GB


In [ ]:
# =========================================================
# 6. INGEST BALANCED DATASET
# =========================================================

print(
    f"\n📥 Ingesting Training Dataset from:\n"
    f"{BALANCED_CSV_PATH}"
)

df = (
    pd.read_csv(BALANCED_CSV_PATH)
    .dropna(
        subset=[
            "cefr",
            "clean_text",
            "topic_title"
        ]
    )
)

df["cefr"] = (
    df["cefr"]
    .str.upper()
    .str.strip()
)


label_map = {
    "A1": 0,
    "A2": 1,
    "B1": 2,
    "B2": 3,
    "C1": 4,
    "C2": 5
}


df["cefr_id"] = (
    df["cefr"].map(label_map)
)

df = df.dropna(
    subset=["cefr_id"]
)

df["cefr_id"] = (
    df["cefr_id"].astype(int)
)


# ---------------------------------------------------------
# EXACT SAME 90/10 STRATIFIED SPLIT
# ---------------------------------------------------------

train_df, val_df = train_test_split(
    df,
    test_size=0.10,
    stratify=df["cefr_id"],
    random_state=42
)


print(
    f"Training partition size:   "
    f"{len(train_df):,} samples"
)

print(
    f"Validation partition size: "
    f"{len(val_df):,} samples"
)


📥 Ingesting Training Dataset from:
/content/drive/MyDrive/Mohammd_Thesis/subsets/efcamdat_balanced_subset_training.csv
Training partition size:   5,011 samples
Validation partition size: 557 samples


In [ ]:
# =========================================================
# 7. LOAD FROZEN BASE LLM & TOKENIZER
# =========================================================

model_id = (
    "meta-llama/"
    "Llama-3.1-8B-Instruct"
)

print(
    f"\nLoading tokenizer and base model: "
    f"{model_id}..."
)


# ---------------------------------------------------------
# Tokenizer
# ---------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    model_id
)

tokenizer.padding_side = "right"

if tokenizer.pad_token is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )

    tokenizer.pad_token_id = (
        tokenizer.eos_token_id
    )


# ---------------------------------------------------------
# Frozen backbone
# ---------------------------------------------------------

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map={
        "": torch.cuda.current_device()
    }
)


for param in base_model.parameters():

    param.requires_grad = False


base_model.eval()

base_model.config.use_cache = False


Loading tokenizer and base model: meta-llama/Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
# =========================================================
# 8. CEFR MULTI-PREFIX CONTROLLER — ~68M
# =========================================================
#
# Llama-3.1-8B:
#
# hidden_size        = 4096
# attention_heads    = 32
# KV heads           = 8
# head_dim           = 128
#
# Therefore:
#
# KV token dimension:
#
#     8 * 128 = 1024
#
# Prefix output dimension:
#
#     32 layers
#     * 2 (K + V)
#     * 8 KV heads
#     * 128 head_dim
#
#     = 65,536
#
# Standard PEFT Prefix-Tuning effectively uses:
#
#     1024 -> 1024 -> 65,536
#
# We reproduce that dimensionality here, but retain
# SIX independent CEFR-specific prefix embedding banks.
#
# =========================================================

class CEFRMultiPrefixController(nn.Module):

    """
    Parameter-matched CEFR Multi-Prefix Tuning controller.

    The controller preserves the proposed CEFR knob:

        A1 -> dedicated 30-token prefix
        A2 -> dedicated 30-token prefix
        B1 -> dedicated 30-token prefix
        B2 -> dedicated 30-token prefix
        C1 -> dedicated 30-token prefix
        C2 -> dedicated 30-token prefix

    A single shared MLP transforms the selected prefix
    into layer-wise K/V states.

    Prefix representation width is 1024, matching the
    Llama-3.1-8B GQA key/value dimension and the
    parameterization used by the Standard PEFT
    Prefix-Tuning baseline.
    """

    def __init__(
        self,
        config,
        num_virtual_tokens=30,
        num_classes=6
    ):

        super().__init__()

        self.num_virtual_tokens = (
            num_virtual_tokens
        )

        self.num_classes = (
            num_classes
        )

        self.num_layers = (
            config.num_hidden_layers
        )

        self.hidden_size = (
            config.hidden_size
        )

        self.num_attention_heads = (
            config.num_attention_heads
        )

        self.num_kv_heads = (
            config.num_key_value_heads
        )

        self.head_dim = getattr(
            config,
            "head_dim",
            (
                config.hidden_size
                // config.num_attention_heads
            )
        )


        # =================================================
        # CRITICAL PARAMETER-MATCHING DIMENSION
        # =================================================
        #
        # 8 KV heads * 128 dimensions = 1024
        #
        # This replaces the 4096-dimensional representation
        # used in the previous 286M CEFR PT experiment.
        #
        # =================================================

        self.token_dim = (
            self.num_kv_heads
            * self.head_dim
        )


        # =================================================
        # CEFR-SPECIFIC PREFIX BANKS
        # =================================================
        #
        # 6 classes
        # * 30 virtual tokens
        # * 1024 dimensions
        #
        # = 184,320 parameters
        #
        # =================================================

        self.prefix_embeddings = nn.Embedding(
            (
                num_classes
                * num_virtual_tokens
            ),
            self.token_dim
        )


        # =================================================
        # PREFIX OUTPUT DIMENSION
        # =================================================
        #
        # 32 * 2 * 8 * 128
        #
        # = 65,536
        #
        # =================================================

        flat_out_dim = (
            self.num_layers
            * 2
            * self.num_kv_heads
            * self.head_dim
        )

        self.flat_out_dim = (
            flat_out_dim
        )


        # =================================================
        # SHARED MLP REPARAMETERIZATION
        # =================================================
        #
        # MATCHED TO STANDARD PEFT PREFIX-TUNING:
        #
        #     1024
        #       ↓
        #     1024
        #       ↓
        #    65536
        #
        # =================================================

        self.prefix_mlp = nn.Sequential(

            nn.Linear(
                self.token_dim,
                self.token_dim
            ),

            nn.Tanh(),

            nn.Linear(
                self.token_dim,
                flat_out_dim
            )
        )


    def forward(
        self,
        cefr_ids
    ):

        batch_size = (
            cefr_ids.shape[0]
        )


        # =================================================
        # 1. SELECT CEFR-SPECIFIC PREFIX
        # =================================================

        class_offsets = (
            cefr_ids
            * self.num_virtual_tokens
        ).unsqueeze(1)


        base_indices = torch.arange(
            self.num_virtual_tokens,
            device=cefr_ids.device
        ).unsqueeze(0)


        token_indices = (
            class_offsets
            + base_indices
        )


        # -------------------------------------------------
        # Shape:
        #
        # [B, 30, 1024]
        # -------------------------------------------------

        prefix_tokens = (
            self.prefix_embeddings(
                token_indices
            )
        )


        # =================================================
        # 2. SHARED MLP PROJECTION
        # =================================================
        #
        # [B, 30, 1024]
        #
        # ->
        #
        # [B, 30, 65536]
        #
        # =================================================

        past_kv_flat = (
            self.prefix_mlp(
                prefix_tokens
            )
        )


        # =================================================
        # 3. RESHAPE INTO LLAMA GQA K/V STATES
        # =================================================
        #
        # [B,
        #  30,
        #  32 layers,
        #  2 (K/V),
        #  8 KV heads,
        #  128 head dim]
        #
        # =================================================

        past_kv = past_kv_flat.view(
            batch_size,
            self.num_virtual_tokens,
            self.num_layers,
            2,
            self.num_kv_heads,
            self.head_dim
        )


        # =================================================
        # 4. PERMUTE
        # =================================================
        #
        # ->
        #
        # [layers,
        #  2,
        #  B,
        #  KV heads,
        #  virtual tokens,
        #  head dim]
        #
        # =================================================

        past_kv = past_kv.permute(
            2,
            3,
            0,
            4,
            1,
            5
        )


        # =================================================
        # 5. BUILD DYNAMIC CACHE
        # =================================================

        past_key_values = (
            DynamicCache()
        )


        for layer_idx in range(
            self.num_layers
        ):

            past_key_values.update(

                past_kv[
                    layer_idx,
                    0
                ],

                past_kv[
                    layer_idx,
                    1
                ],

                layer_idx=layer_idx
            )


        return past_key_values


# =========================================================
# 9. INITIALIZE CONTROLLER
# =========================================================

prefix_controller = (
    CEFRMultiPrefixController(
        base_model.config,
        num_virtual_tokens=30,
        num_classes=6
    )
    .to(
        device,
        dtype=torch.bfloat16
    )
)


# =========================================================
# 10. PARAMETER-MATCHING VERIFICATION
# =========================================================

trainable_params = sum(
    p.numel()
    for p in prefix_controller.parameters()
    if p.requires_grad
)


STANDARD_PT_REFERENCE_PARAMS = (
    68_254_720
)


parameter_difference = (
    trainable_params
    - STANDARD_PT_REFERENCE_PARAMS
)


parameter_difference_pct = (
    parameter_difference
    / STANDARD_PT_REFERENCE_PARAMS
    * 100
)


print("\n" + "=" * 70)

print(
    "🔢 PARAMETER-MATCHING VERIFICATION"
)

print("=" * 70)


print(
    f"Llama hidden size              : "
    f"{base_model.config.hidden_size}"
)

print(
    f"Number of attention heads      : "
    f"{base_model.config.num_attention_heads}"
)

print(
    f"Number of KV heads             : "
    f"{base_model.config.num_key_value_heads}"
)

print(
    f"Head dimension                 : "
    f"{prefix_controller.head_dim}"
)

print(
    f"Prefix token dimension         : "
    f"{prefix_controller.token_dim}"
)

print(
    f"Prefix projection output dim   : "
    f"{prefix_controller.flat_out_dim}"
)

print("-" * 70)

print(
    f"Standard PT parameters         : "
    f"{STANDARD_PT_REFERENCE_PARAMS:,}"
)

print(
    f"CEFR PT parameters             : "
    f"{trainable_params:,}"
)

print(
    f"Difference                     : "
    f"{parameter_difference:+,}"
)

print(
    f"Difference percentage          : "
    f"{parameter_difference_pct:+.4f}%"
)

print("=" * 70)


# ---------------------------------------------------------
# Expected:
#
# Standard PT:
#   68,254,720
#
# CEFR PT:
#   68,408,320
#
# Difference:
#   +153,600
#
# ~0.225%
# ---------------------------------------------------------

assert abs(
    parameter_difference_pct
) < 0.5, (
    "Parameter matching failed: "
    "difference exceeds 0.5%."
)


print(
    "✅ CEFR Prefix-Tuning is parameter-matched "
    "to Standard Prefix-Tuning."
)



🔢 PARAMETER-MATCHING VERIFICATION
Llama hidden size              : 4096
Number of attention heads      : 32
Number of KV heads             : 8
Head dimension                 : 128
Prefix token dimension         : 1024
Prefix projection output dim   : 65536
----------------------------------------------------------------------
Standard PT parameters         : 68,254,720
CEFR PT parameters             : 68,408,320
Difference                     : +153,600
Difference percentage          : +0.2250%
✅ CEFR Prefix-Tuning is parameter-matched to Standard Prefix-Tuning.


In [ ]:
# =========================================================
# 11. DATASET UTILITY — IDENTICAL BLIND PROMPT
# =========================================================

class CEFRTopicDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length=512
    ):

        self.data = (
            dataframe
            .reset_index(drop=True)
        )

        self.tokenizer = tokenizer

        self.max_length = (
            max_length
        )


    def __len__(self):

        return len(
            self.data
        )


    def __getitem__(
        self,
        idx
    ):

        row = (
            self.data.iloc[idx]
        )

        topic_title = str(
            row["topic_title"]
        ).strip()


        # =================================================
        # EXACT SAME BLIND PROMPT AS PREVIOUS CEFR PT
        # =================================================
        #
        # Explicit target CEFR label is NOT included.
        #
        # A1/A2/B1/B2/C1/C2 is supplied exclusively
        # through prefix selection.
        #
        # =================================================

        prompt = (

            f"You are an expert English language teacher "
            f"demonstrating CEFR proficiency levels. "

            f"Your task is to write a flawless, "
            f"grammatically correct text responding to "
            f"this prompt: '{topic_title}'. "

            f"If the requested target level is A1/A2, "
            f"use very simple vocabulary, short sentences, "
            f"and primitive structures. "

            f"If the requested target level is C1/C2, "
            f"utilize highly advanced vocabulary, idioms, "
            f"and complex sentence patterns. "

            f"Write only the direct response. "
            f"Do not write any meta-commentary, greetings, "
            f"or conversational pleasantries."
        )


        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]


        formatted_prompt = (
            self.tokenizer
            .apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        )


        full_text = (
            formatted_prompt
            + str(
                row["clean_text"]
            ).strip()
            + self.tokenizer.eos_token
        )


        # =================================================
        # PROMPT TOKENIZATION
        # =================================================

        prompt_enc = self.tokenizer(
            formatted_prompt,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length
        )


        prompt_len = len(
            prompt_enc["input_ids"]
        )


        # =================================================
        # FULL SEQUENCE TOKENIZATION
        # =================================================

        full_enc = self.tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
            padding="max_length"
        )


        input_ids = torch.tensor(
            full_enc["input_ids"],
            dtype=torch.long
        )


        attention_mask = torch.tensor(
            full_enc["attention_mask"],
            dtype=torch.long
        )


        # =================================================
        # LOSS MASKING
        # =================================================

        labels = input_ids.clone()


        # Do not train on prompt tokens
        labels[
            :prompt_len
        ] = -100


        # Do not train on padding
        labels[
            attention_mask == 0
        ] = -100


        return {

            "input_ids":
                input_ids,

            "attention_mask":
                attention_mask,

            "labels":
                labels,

            "cefr_id":
                torch.tensor(
                    row["cefr_id"],
                    dtype=torch.long
                )
        }


In [ ]:
# =========================================================
# 12. DATA LOADERS
# =========================================================
#
# EXACT SAME SETTINGS AS 286M CEFR PT
#
# Effective batch:
#
#     16 * 2 = 32
#
# =========================================================

TRAIN_BATCH_SIZE = 16

GRADIENT_ACCUMULATION_STEPS = 2

VAL_BATCH_SIZE = 32


train_dataset = CEFRTopicDataset(
    train_df,
    tokenizer
)


val_dataset = CEFRTopicDataset(
    val_df,
    tokenizer
)


train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)


print("\n📦 DataLoader configuration")

print(
    f"Training batch size       : "
    f"{TRAIN_BATCH_SIZE}"
)

print(
    f"Gradient accumulation     : "
    f"{GRADIENT_ACCUMULATION_STEPS}"
)

print(
    f"Effective batch size      : "
    f"{TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}"
)

print(
    f"Validation batch size     : "
    f"{VAL_BATCH_SIZE}"
)


# =========================================================
# 13. HARDWARE PROFILER
# =========================================================

class GPUMonitor(
    threading.Thread
):

    def __init__(
        self,
        delay=1.0
    ):

        super(
            GPUMonitor,
            self
        ).__init__()

        self.stopped = False

        self.delay = delay

        self.utilization_rates = []

        pynvml.nvmlInit()

        self.handle = (
            pynvml
            .nvmlDeviceGetHandleByIndex(0)
        )


    def run(self):

        while not self.stopped:

            try:

                util = (
                    pynvml
                    .nvmlDeviceGetUtilizationRates(
                        self.handle
                    )
                )

                self.utilization_rates.append(
                    util.gpu
                )

            except Exception:

                pass

            time.sleep(
                self.delay
            )


    def stop(self):

        self.stopped = True

        try:

            pynvml.nvmlShutdown()

        except Exception:

            pass


    def get_avg_utilization(self):

        if not self.utilization_rates:

            return 0.0

        return (
            sum(
                self.utilization_rates
            )
            / len(
                self.utilization_rates
            )
        )


# =========================================================
# 14. OPTIMIZATION CONFIGURATION
# =========================================================
#
# EXACT SAME SETTINGS AS PREVIOUS CEFR PT
#
# =========================================================

epochs = 3

lr = 2e-4


optimizer = torch.optim.AdamW(
    prefix_controller.parameters(),
    lr=lr,
    weight_decay=0.01
)


total_training_steps = (
    math.ceil(
        len(train_loader)
        / GRADIENT_ACCUMULATION_STEPS
    )
    * epochs
)


warmup_steps = int(
    0.05
    * total_training_steps
)


scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps
    )
)


best_val_loss = float(
    "inf"
)


training_summary_logs = []


print(
    f"\n🚀 Launching Parameter-Matched "
    f"CEFR Multi-Prefix Tuning..."
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,} "
    f"({trainable_params / 1e6:.3f}M)"
)


# =========================================================
# 15. START HARDWARE PROFILING
# =========================================================

torch.cuda.reset_peak_memory_stats()


gpu_monitor = GPUMonitor(
    delay=1.0
)

gpu_monitor.start()


start_time = (
    time.perf_counter()
)


📦 DataLoader configuration
Training batch size       : 16
Gradient accumulation     : 2
Effective batch size      : 32
Validation batch size     : 32

🚀 Launching Parameter-Matched CEFR Multi-Prefix Tuning...
Trainable parameters: 68,408,320 (68.408M)


In [ ]:
# =========================================================
# 16. TRAINING LOOP
# =========================================================

for epoch in range(
    epochs
):


    # =====================================================
    # TRAINING PHASE
    # =====================================================

    prefix_controller.train()


    running_train_loss = (
        0.0
    )


    optimizer.zero_grad()


    progress_bar = tqdm(

        train_loader,

        desc=(
            f"Epoch "
            f"{epoch + 1}/{epochs} "
            f"[Train]"
        ),

        leave=False
    )


    for step, batch in enumerate(
        progress_bar
    ):


        input_ids = (
            batch["input_ids"]
            .to(device)
        )


        attention_mask = (
            batch["attention_mask"]
            .to(device)
        )


        labels = (
            batch["labels"]
            .to(device)
        )


        cefr_ids = (
            batch["cefr_id"]
            .to(device)
        )


        # =================================================
        # 1. CEFR-SPECIFIC PREFIX CACHE
        # =================================================

        past_key_values = (
            prefix_controller(
                cefr_ids
            )
        )


        # =================================================
        # 2. PREFIX ATTENTION MASK
        # =================================================

        prefix_mask = torch.ones(

            input_ids.shape[0],

            prefix_controller
            .num_virtual_tokens,

            dtype=(
                attention_mask.dtype
            ),

            device=device
        )


        full_attention_mask = (
            torch.cat(
                [
                    prefix_mask,
                    attention_mask
                ],
                dim=1
            )
        )


        # =================================================
        # 3. POSITION IDS
        # =================================================
        #
        # EXACT SAME LOGIC AS PREVIOUS CEFR PT.
        #
        # =================================================

        position_ids = (
            full_attention_mask
            .long()
            .cumsum(-1)
            - 1
        )


        position_ids.masked_fill_(
            full_attention_mask == 0,
            1
        )


        position_ids = position_ids[
            :,
            prefix_controller
            .num_virtual_tokens:
        ]


        # =================================================
        # 4. FROZEN LLAMA FORWARD PASS
        # =================================================

        outputs = base_model(

            input_ids=input_ids,

            attention_mask=(
                full_attention_mask
            ),

            position_ids=(
                position_ids
            ),

            past_key_values=(
                past_key_values
            ),

            labels=labels
        )


        loss = outputs.loss


        # =================================================
        # 5. GRADIENT ACCUMULATION
        # =================================================

        loss_for_backward = (
            loss
            / GRADIENT_ACCUMULATION_STEPS
        )


        loss_for_backward.backward()


        running_train_loss += (
            loss.item()
        )


        # =================================================
        # 6. OPTIMIZER STEP
        # =================================================

        if (
            (
                (step + 1)
                % GRADIENT_ACCUMULATION_STEPS
                == 0
            )
            or
            (
                (step + 1)
                == len(train_loader)
            )
        ):


            torch.nn.utils.clip_grad_norm_(
                prefix_controller.parameters(),
                max_norm=1.0
            )


            optimizer.step()


            scheduler.step()


            optimizer.zero_grad()


        progress_bar.set_postfix(
            {
                "Loss":
                    f"{loss.item():.4f}"
            }
        )


    # =====================================================
    # VALIDATION PHASE
    # =====================================================

    prefix_controller.eval()


    val_loss_accum = (
        0.0
    )


    val_steps = 0


    with torch.no_grad():


        for batch in tqdm(

            val_loader,

            desc=(
                f"Epoch "
                f"{epoch + 1}/{epochs} "
                f"[Val]"
            ),

            leave=False
        ):


            input_ids = (
                batch["input_ids"]
                .to(device)
            )


            attention_mask = (
                batch["attention_mask"]
                .to(device)
            )


            labels = (
                batch["labels"]
                .to(device)
            )


            cefr_ids = (
                batch["cefr_id"]
                .to(device)
            )


            # ---------------------------------------------
            # CEFR-specific prefix
            # ---------------------------------------------

            past_key_values = (
                prefix_controller(
                    cefr_ids
                )
            )


            # ---------------------------------------------
            # Attention mask
            # ---------------------------------------------

            prefix_mask = torch.ones(

                input_ids.shape[0],

                prefix_controller
                .num_virtual_tokens,

                dtype=(
                    attention_mask.dtype
                ),

                device=device
            )


            full_attention_mask = (
                torch.cat(
                    [
                        prefix_mask,
                        attention_mask
                    ],
                    dim=1
                )
            )


            # ---------------------------------------------
            # Position IDs
            # ---------------------------------------------

            position_ids = (
                full_attention_mask
                .long()
                .cumsum(-1)
                - 1
            )


            position_ids.masked_fill_(
                full_attention_mask == 0,
                1
            )


            position_ids = position_ids[
                :,
                prefix_controller
                .num_virtual_tokens:
            ]


            # ---------------------------------------------
            # Forward
            # ---------------------------------------------

            outputs = base_model(

                input_ids=input_ids,

                attention_mask=(
                    full_attention_mask
                ),

                position_ids=(
                    position_ids
                ),

                past_key_values=(
                    past_key_values
                ),

                labels=labels
            )


            val_loss_accum += (
                outputs.loss.item()
            )


            val_steps += 1


    # =====================================================
    # EPOCH METRICS
    # =====================================================

    avg_train_loss = (
        running_train_loss
        / len(train_loader)
    )


    avg_val_loss = (
        val_loss_accum
        / val_steps
    )


    val_ppl = float(
        np.exp(
            avg_val_loss
        )
    )


    epoch_metrics = {

        "epoch":
            epoch + 1,

        "train_loss":
            round(
                avg_train_loss,
                4
            ),

        "val_loss":
            round(
                avg_val_loss,
                4
            ),

        "val_ppl":
            round(
                val_ppl,
                2
            )
    }


    training_summary_logs.append(
        epoch_metrics
    )


    print(

        f"\n📈 "
        f"[Epoch {epoch + 1}/{epochs}] "
        f"Done | "

        f"Train Loss: "
        f"{avg_train_loss:.4f} | "

        f"Val Loss: "
        f"{avg_val_loss:.4f} | "

        f"Val PPL: "
        f"{val_ppl:.2f}"
    )


    # =====================================================
    # SAVE BEST CHECKPOINT
    # =====================================================

    if (
        avg_val_loss
        < best_val_loss
    ):


        best_val_loss = (
            avg_val_loss
        )


        checkpoint_path = (
            os.path.join(

                LOCAL_OUTPUT_DIR,

                "cefr_prefix_tuning_68m_best_weights.pt"
            )
        )


        torch.save(
            prefix_controller.state_dict(),
            checkpoint_path
        )


        print(
            f"🌟 Best validation score achieved!"
        )


        print(
            f"Saved checkpoint to:\n"
            f"{checkpoint_path}"
        )


    torch.cuda.empty_cache()

    gc.collect()


# =========================================================
# 17. STOP HARDWARE PROFILER
# =========================================================

gpu_monitor.stop()


end_time = (
    time.perf_counter()
)


total_time_seconds = (
    end_time
    - start_time
)


total_time_hours = (
    total_time_seconds
    / 3600
)


avg_gpu_util = (
    gpu_monitor
    .get_avg_utilization()
)


peak_vram_gb = (
    torch.cuda
    .max_memory_allocated()
    / (1024 ** 3)
)

Epoch 1/3 [Train]:   0%|          | 0/314 [00:00<?, ?it/s]

Epoch 1/3 [Val]:   0%|          | 0/18 [00:00<?, ?it/s]


📈 [Epoch 1/3] Done | Train Loss: 2.7607 | Val Loss: 2.5314 | Val PPL: 12.57
🌟 Best validation score achieved!
Saved checkpoint to:
./cefr_prefix_tuning_68m_param_matched_export/cefr_prefix_tuning_68m_best_weights.pt


Epoch 2/3 [Train]:   0%|          | 0/314 [00:00<?, ?it/s]

Epoch 2/3 [Val]:   0%|          | 0/18 [00:00<?, ?it/s]


📈 [Epoch 2/3] Done | Train Loss: 2.4281 | Val Loss: 2.4825 | Val PPL: 11.97
🌟 Best validation score achieved!
Saved checkpoint to:
./cefr_prefix_tuning_68m_param_matched_export/cefr_prefix_tuning_68m_best_weights.pt


Epoch 3/3 [Train]:   0%|          | 0/314 [00:00<?, ?it/s]

Epoch 3/3 [Val]:   0%|          | 0/18 [00:00<?, ?it/s]


📈 [Epoch 3/3] Done | Train Loss: 2.3348 | Val Loss: 2.4855 | Val PPL: 12.01


In [ ]:
# =========================================================
# 18. SAVE TRAINING SUMMARY
# =========================================================

training_summary_filename = (
    "cefr_prefix_tuning_68m_"
    "parameter_matched_training_summary.json"
)


drive_summary_path = (
    os.path.join(
        DRIVE_LOG_DIR,
        training_summary_filename
    )
)


with open(
    drive_summary_path,
    "w"
) as f:

    json.dump(
        training_summary_logs,
        f,
        indent=4
    )


local_summary_path = (
    os.path.join(
        LOCAL_OUTPUT_DIR,
        training_summary_filename
    )
)


with open(
    local_summary_path,
    "w"
) as f:

    json.dump(
        training_summary_logs,
        f,
        indent=4
    )


# =========================================================
# 19. HARDWARE PROFILING REPORT
# =========================================================

profiling_report = (

    "=== CEFR Multi-Prefix Tuning "
    "68M Parameter-Matched Hardware Report ===\n"

    f"Total training time: "
    f"{total_time_seconds:.2f} seconds "
    f"({total_time_hours:.2f} hours)\n"

    f"Peak GPU memory: "
    f"{peak_vram_gb:.2f} GB\n"

    f"Average GPU utilization: "
    f"{avg_gpu_util:.1f}%\n"

    f"Trainable parameters: "
    f"{trainable_params:,}\n"

    f"Standard PT reference parameters: "
    f"{STANDARD_PT_REFERENCE_PARAMS:,}\n"

    f"Parameter difference: "
    f"{parameter_difference:+,} "
    f"({parameter_difference_pct:+.4f}%)\n"
)


print(
    "\n"
    + profiling_report
)


profiling_path = (
    os.path.join(
        DRIVE_LOG_DIR,
        "cefr_prefix_tuning_68m_profiling.txt"
    )
)


with open(
    profiling_path,
    "w"
) as f:

    f.write(
        profiling_report
    )


# =========================================================
# 20. SAVE EXPERIMENT METADATA
# =========================================================

metadata = {

    "experiment":
        "CEFR Multi-Prefix Tuning 68M Parameter-Matched",

    "base_model":
        model_id,

    "dataset":
        BALANCED_CSV_PATH,

    "trainable_parameters":
        trainable_params,

    "standard_prefix_tuning_reference_parameters":
        STANDARD_PT_REFERENCE_PARAMS,

    "parameter_difference":
        parameter_difference,

    "parameter_difference_percent":
        parameter_difference_pct,

    "num_cefr_classes":
        6,

    "num_virtual_tokens_per_class":
        30,

    "total_cefr_prefix_embeddings":
        180,

    "prefix_token_dimension":
        prefix_controller.token_dim,

    "prefix_projection_output_dimension":
        prefix_controller.flat_out_dim,

    "mlp_architecture":
        (
            f"{prefix_controller.token_dim}"
            f" -> "
            f"{prefix_controller.token_dim}"
            f" -> "
            f"{prefix_controller.flat_out_dim}"
        ),

    "explicit_cefr_in_textual_prompt":
        False,

    "conditioning":
        "CEFR-specific prefix-bank selection",

    "epochs":
        epochs,

    "learning_rate":
        lr,

    "train_batch_size":
        TRAIN_BATCH_SIZE,

    "gradient_accumulation_steps":
        GRADIENT_ACCUMULATION_STEPS,

    "effective_batch_size":
        (
            TRAIN_BATCH_SIZE
            * GRADIENT_ACCUMULATION_STEPS
        ),

    "validation_batch_size":
        VAL_BATCH_SIZE,

    "seed":
        SEED
}


metadata_path = (
    os.path.join(
        LOCAL_OUTPUT_DIR,
        "experiment_metadata.json"
    )
)


with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )


# =========================================================
# 21. SAVE TOKENIZER
# =========================================================

tokenizer.save_pretrained(
    LOCAL_OUTPUT_DIR
)


# =========================================================
# 22. GENERATE README
# =========================================================

readme_content = f"""---
license: apache-2.0
base_model: {model_id}
tags:
- prefix-tuning
- cefr-control
- cefr-conditioning
- multi-prefix
- parameter-matched
- ablation
- topic-aligned
---

# CEFR Multi-Prefix Tuning — 68M Parameter-Matched Control

This repository contains the parameter-matched CEFR-conditioned
Multi-Prefix Tuning experiment based on Llama-3.1-8B-Instruct.

## Experimental Purpose

This model was constructed as a controlled comparison against
the Standard PEFT Prefix-Tuning baseline.

The Standard Prefix-Tuning baseline contains:

- **{STANDARD_PT_REFERENCE_PARAMS:,} trainable parameters**

This CEFR-conditioned model contains:

- **{trainable_params:,} trainable parameters**

The difference is:

- **{parameter_difference:+,} parameters**
- **{parameter_difference_pct:+.4f}%**

This allows the effect of explicit CEFR-specific prefix
conditioning to be studied while keeping the trainable
parameter budget approximately constant.

## Architecture

Llama-3.1-8B-Instruct uses Grouped Query Attention (GQA).

For the prefix representation:

- KV heads: {prefix_controller.num_kv_heads}
- Head dimension: {prefix_controller.head_dim}
- Prefix token dimension: {prefix_controller.token_dim}

The shared prefix projection is:

**{prefix_controller.token_dim} -> {prefix_controller.token_dim} -> {prefix_controller.flat_out_dim}**

This matches the representation width used by the Standard
Prefix-Tuning baseline.

## CEFR Conditioning

The model contains six independent CEFR-specific prefix banks:

- A1
- A2
- B1
- B2
- C1
- C2

Each CEFR level contains 30 learned virtual tokens.

Therefore:

- 6 CEFR levels
- 30 virtual tokens per level
- 180 total CEFR-specific virtual-token embeddings

At training and inference time, the requested CEFR class
selects the corresponding 30-token prefix.

The projection MLP is shared across all six CEFR levels.

## Textual Prompt

The textual instruction is blind with respect to the specific
target CEFR class.

The target label (A1--C2) is not explicitly inserted into the
prompt. The requested proficiency level is communicated through
the selected CEFR-specific prefix bank.

## Training Configuration

- Dataset: balanced 6k EFCAMDAT training subset
- Split: stratified 90/10
- Random seed: {SEED}
- Epochs: {epochs}
- Learning rate: {lr}
- Optimizer: AdamW
- Weight decay: 0.01
- Scheduler: cosine
- Warmup: 5%
- Training batch size: {TRAIN_BATCH_SIZE}
- Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}
- Effective batch size: {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}
- Validation batch size: {VAL_BATCH_SIZE}
- Maximum sequence length: 512
- Backbone: frozen
- Precision: BF16

## Training Results

{json.dumps(training_summary_logs, indent=2)}

## Hardware Profiling

- Training time: {total_time_seconds:.2f} seconds ({total_time_hours:.2f} hours)
- Peak GPU memory: {peak_vram_gb:.2f} GB
- Average GPU utilization: {avg_gpu_util:.1f}%

## Role in Thesis

This experiment is intended to isolate the contribution of the
explicit CEFR conditioning mechanism.

It should be compared directly against:

1. Standard Prefix-Tuning (~68.25M)
2. CEFR Multi-Prefix Tuning (~68.41M)

The larger CEFR Prefix-Tuning experiments (~286M and ~537M)
are used separately to study controller-capacity scaling and
the parameter-matched comparison with CEFR-gated PMT.
"""


readme_path = (
    os.path.join(
        LOCAL_OUTPUT_DIR,
        "README.md"
    )
)


with open(
    readme_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        readme_content
    )


print(
    f"\n✅ README saved to:\n"
    f"{readme_path}"
)


# =========================================================
# 23. EXPORT TO HUGGING FACE
# =========================================================

print(
    "\n📦 Pushing CEFR 68M parameter-matched "
    "controller to Hugging Face..."
)


try:

    api = HfApi()


    api.create_repo(
        repo_id=HF_REPO_ID,
        repo_type="model",
        exist_ok=True
    )


    api.upload_folder(
        folder_path=LOCAL_OUTPUT_DIR,
        repo_id=HF_REPO_ID,
        repo_type="model",
        commit_message=(
            "Upload CEFR Multi-Prefix Tuning "
            f"68M parameter-matched weights "
            f"(Val PPL: {np.exp(best_val_loss):.2f})"
        )
    )


    print(
        "\n🚀 SUCCESS!"
    )


    print(
        f"Model deployed to:\n"
        f"https://huggingface.co/{HF_REPO_ID}"
    )


except Exception as e:

    print(
        f"⚠️ Export failed: "
        f"{str(e)}"
    )


# =========================================================
# 24. FINAL EXPERIMENT SUMMARY
# =========================================================

print(
    "\n"
    + "=" * 70
)

print(
    "FINAL EXPERIMENT SUMMARY"
)

print(
    "=" * 70
)


print(
    f"Standard PT reference  : "
    f"{STANDARD_PT_REFERENCE_PARAMS:,} "
    f"({STANDARD_PT_REFERENCE_PARAMS / 1e6:.3f}M)"
)


print(
    f"CEFR PT trainable      : "
    f"{trainable_params:,} "
    f"({trainable_params / 1e6:.3f}M)"
)


print(
    f"Difference             : "
    f"{parameter_difference:+,}"
)


print(
    f"Difference percentage  : "
    f"{parameter_difference_pct:+.4f}%"
)


print(
    f"Prefix token dimension : "
    f"{prefix_controller.token_dim}"
)


print(
    f"MLP architecture       : "
    f"{prefix_controller.token_dim}"
    f" -> "
    f"{prefix_controller.token_dim}"
    f" -> "
    f"{prefix_controller.flat_out_dim}"
)


print(
    f"HF repository          : "
    f"{HF_REPO_ID}"
)


print(
    f"Drive log directory    : "
    f"{DRIVE_LOG_DIR}"
)


print(
    "=" * 70
)


print(
    "\n✅ 68M CEFR parameter-matched "
    "Prefix-Tuning experiment completed."
)


=== CEFR Multi-Prefix Tuning 68M Parameter-Matched Hardware Report ===
Total training time: 1489.85 seconds (0.41 hours)
Peak GPU memory: 67.80 GB
Average GPU utilization: 97.6%
Trainable parameters: 68,408,320
Standard PT reference parameters: 68,254,720
Parameter difference: +153,600 (+0.2250%)


✅ README saved to:
./cefr_prefix_tuning_68m_param_matched_export/README.md

📦 Pushing CEFR 68M parameter-matched controller to Hugging Face...

🚀 SUCCESS!
Model deployed to:
https://huggingface.co/MohammadKhosravi/llama3.1-8b-cefr-prefix-tuning-68m-param-matched

FINAL EXPERIMENT SUMMARY
Standard PT reference  : 68,254,720 (68.255M)
CEFR PT trainable      : 68,408,320 (68.408M)
Difference             : +153,600
Difference percentage  : +0.2250%
Prefix token dimension : 1024
MLP architecture       : 1024 -> 1024 -> 65536
HF repository          : MohammadKhosravi/llama3.1-8b-cefr-prefix-tuning-68m-param-matched
Drive log directory    : /content/drive/MyDrive/Mohammd_Thesis/Training_Logs/CEFR_

## Generation and Evaluation

This section loads the trained 68M CEFR Multi-Prefix controller and evaluates it on the final In-Domain Evaluation Prompt Matrix.

During generation, the textual prompt does not explicitly contain the target CEFR label. Instead, the target level is supplied through the CEFR-specific prefix bank selected by the controller.

Generated texts are evaluated using the primary Joint-Loss CEFR evaluator and the same linguistic diagnostics used for the other thesis methods.

In [ ]:
# =========================================================
# 0. ENVIRONMENT SETUP & DEPENDENCIES
# =========================================================
!pip install -q transformers torch datasets tqdm huggingface_hub accelerate scikit-learn textstat spacy hf_transfer "torchao>=0.16.0"
!python -m spacy download en_core_web_sm

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import gc
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import spacy
import textstat
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, set_seed
from transformers.cache_utils import DynamicCache
from huggingface_hub import login, hf_hub_download
from google.colab import drive, userdata
from tqdm.auto import tqdm

# =========================================================
# 1. REPRODUCIBILITY
# =========================================================
SEED = 42
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# 2. MOUNT GOOGLE DRIVE
# =========================================================
drive.mount("/content/drive")

# =========================================================
# 3. EXPERIMENT PATHS
# =========================================================
INPUT_CSV = "/content/drive/MyDrive/Your_Path/"
    "in_domain_evaluation_prompt_matrix.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Your_Path/"
    "cefr_prefix_tuning_68m_evaluation_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV_PATH = os.path.join(OUTPUT_DIR, "cefr_prefix_68m_param_matched_benchmark_results_log.csv")
OUTPUT_TXT_PATH = os.path.join(OUTPUT_DIR, "cefr_prefix_68m_param_matched_overall_metrics_report.txt")
OUTPUT_IMG_PATH = os.path.join(OUTPUT_DIR, "cefr_prefix_68m_param_matched_confusion_matrix.png")

# =========================================================
# 4. HUGGING FACE LOGIN
# =========================================================
try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("✔️ Hugging Face authentication successful.")
except Exception as e:
    print(f"⚠️ Hugging Face login skipped: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 100.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 144.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Mounted at /content/drive
✔️ Hugging Face authentication successful.


In [ ]:
# =========================================================
# 5. GLOBAL CONFIGURATION
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
MAX_NEW_TOKENS = 200
TEMPERATURE = 0.6

label_map = {"A1": 0, "A2": 1, "B1": 2, "B2": 3, "C1": 4, "C2": 5}
inv_label_map = {value: key for key, value in label_map.items()}

print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# =========================================================
# 6. MODEL CONFIGURATION
# =========================================================
model_id = "meta-llama/Llama-3.1-8B-Instruct"
HF_REPO_ID = "MohammadKhosravi/llama3.1-8b-cefr-prefix-tuning-68m-param-matched"
WEIGHTS_FILENAME = "cefr_prefix_tuning_68m_best_weights.pt"
NUM_VIRTUAL_TOKENS = 30
NUM_CLASSES = 6

print("\n" + "=" * 70)
print("CEFR MULTI-PREFIX TUNING 68M PARAMETER-MATCHED INFERENCE")
print("=" * 70)
print(f"Base Model           : {model_id}")
print(f"Controller Repository: {HF_REPO_ID}")
print(f"Virtual Tokens       : {NUM_VIRTUAL_TOKENS}")
print(f"CEFR Classes         : {NUM_CLASSES}")
print(f"Batch Size           : {BATCH_SIZE}")
print(f"Temperature          : {TEMPERATURE}")
print(f"Max New Tokens       : {MAX_NEW_TOKENS}")
print("=" * 70)

# =========================================================
# 7. LOAD TOKENIZER
# =========================================================
print("\n[1/4] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print("✔️ Tokenizer loaded.")

# =========================================================
# 8. LOAD FROZEN BASE MODEL
# =========================================================
print("\n[2/4] Loading Llama-3.1-8B-Instruct...")
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")
base_model.eval()
base_model.config.use_cache = True
print("✔️ Base model loaded.")

Using device: cuda
GPU: NVIDIA L4

CEFR MULTI-PREFIX TUNING 68M PARAMETER-MATCHED INFERENCE
Base Model           : meta-llama/Llama-3.1-8B-Instruct
Controller Repository: MohammadKhosravi/llama3.1-8b-cefr-prefix-tuning-68m-param-matched
Virtual Tokens       : 30
CEFR Classes         : 6
Batch Size           : 32
Temperature          : 0.6
Max New Tokens       : 200

[1/4] Loading tokenizer...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✔️ Base model loaded.


In [ ]:
# =========================================================
# 9. CEFR MULTI-PREFIX CONTROLLER — EXACT 68M ARCHITECTURE
# =========================================================
class CEFRMultiPrefixController(nn.Module):
    def __init__(self, config, num_virtual_tokens=30, num_classes=6):
        super().__init__()
        self.num_virtual_tokens = num_virtual_tokens
        self.num_classes = num_classes
        self.num_layers = config.num_hidden_layers
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
        self.token_dim = self.num_kv_heads * self.head_dim

        self.prefix_embeddings = nn.Embedding(num_classes * num_virtual_tokens, self.token_dim)

        flat_out_dim = self.num_layers * 2 * self.num_kv_heads * self.head_dim
        self.flat_out_dim = flat_out_dim

        self.prefix_mlp = nn.Sequential(
            nn.Linear(self.token_dim, self.token_dim),
            nn.Tanh(),
            nn.Linear(self.token_dim, flat_out_dim)
        )

    def forward(self, cefr_ids):
        batch_size = cefr_ids.shape[0]
        class_offsets = (cefr_ids * self.num_virtual_tokens).unsqueeze(1)
        base_indices = torch.arange(self.num_virtual_tokens, device=cefr_ids.device).unsqueeze(0)
        token_indices = class_offsets + base_indices

        prefix_tokens = self.prefix_embeddings(token_indices)
        past_kv_flat = self.prefix_mlp(prefix_tokens)

        past_kv = past_kv_flat.view(
            batch_size, self.num_virtual_tokens, self.num_layers, 2, self.num_kv_heads, self.head_dim
        )
        past_kv = past_kv.permute(2, 3, 0, 4, 1, 5)

        past_key_values = DynamicCache()
        for layer_idx in range(self.num_layers):
            past_key_values.update(past_kv[layer_idx, 0], past_kv[layer_idx, 1], layer_idx=layer_idx)

        return past_key_values

# =========================================================
# 10. INITIALIZE CONTROLLER
# =========================================================
prefix_controller = CEFRMultiPrefixController(
    base_model.config, num_virtual_tokens=NUM_VIRTUAL_TOKENS, num_classes=NUM_CLASSES
).to(device, dtype=torch.bfloat16)

In [ ]:
# =========================================================
# 11. VERIFY PARAMETER COUNT
# =========================================================
trainable_params = sum(p.numel() for p in prefix_controller.parameters() if p.requires_grad)
STANDARD_PT_REFERENCE_PARAMS = 68_254_720
parameter_difference = trainable_params - STANDARD_PT_REFERENCE_PARAMS
parameter_difference_pct = (parameter_difference / STANDARD_PT_REFERENCE_PARAMS) * 100

print("\n" + "=" * 70)
print("PARAMETER VERIFICATION")
print("=" * 70)
print(f"Prefix token dimension : {prefix_controller.token_dim}")
print(f"Projection output dim  : {prefix_controller.flat_out_dim}")
print(f"Standard PT reference  : {STANDARD_PT_REFERENCE_PARAMS:,}")
print(f"CEFR PT parameters     : {trainable_params:,}")
print(f"Difference             : {parameter_difference:+,}")
print(f"Difference             : {parameter_difference_pct:+.4f}%")
print("=" * 70)

assert trainable_params == 68_408_320, f"Unexpected controller parameter count: {trainable_params:,}"
print("✔️ 68M controller architecture verified.")


PARAMETER VERIFICATION
Prefix token dimension : 1024
Projection output dim  : 65536
Standard PT reference  : 68,254,720
CEFR PT parameters     : 68,408,320
Difference             : +153,600
Difference             : +0.2250%
✔️ 68M controller architecture verified.


In [ ]:
# =========================================================
# 12. DOWNLOAD TRAINED CONTROLLER
# =========================================================
print(f"\nDownloading controller weights from:\n{HF_REPO_ID}")
try:
    weights_path = hf_hub_download(repo_id=HF_REPO_ID, filename=WEIGHTS_FILENAME)
except Exception as e:
    raise RuntimeError(f"\nCould not download the trained 68M controller.\nOriginal error:\n{e}")
print(f"✔️ Controller checkpoint downloaded:\n{weights_path}")

# =========================================================
# 13. LOAD CONTROLLER WEIGHTS
# =========================================================
state_dict = torch.load(weights_path, map_location=device)
prefix_controller.load_state_dict(state_dict, strict=True)
prefix_controller.eval()
print("✔️ Strict checkpoint match successful.")

# =========================================================
# 14. LOAD BENCHMARK DATASET
# =========================================================
print("\n[3/4] Loading EFCAMDAT Evaluation Prompt Matrix...")
df = pd.read_csv(INPUT_CSV)
df["cefr"] = df["cefr"].astype(str).str.strip().str.upper()
invalid_levels = set(df["cefr"]) - set(label_map.keys())

if invalid_levels:
    raise ValueError(f"Invalid CEFR labels found: {sorted(invalid_levels)}")
print(f"Benchmark conditions: {len(df):,}")

# =========================================================
# 15. BLIND PROMPT — IDENTICAL TO TRAINING
# =========================================================
def build_blind_prompt(topic_title):
    prompt = (
        f"You are an expert English language teacher demonstrating CEFR proficiency levels. "
        f"Your task is to write a flawless, grammatically correct text responding to this prompt: '{topic_title}'. "
        f"If the requested target level is A1/A2, use very simple vocabulary, short sentences, and primitive structures. "
        f"If the requested target level is C1/C2, utilize highly advanced vocabulary, idioms, and complex sentence patterns. "
        f"Write only the direct response. Do not write any meta-commentary, greetings, or conversational pleasantries."
    )
    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


MohammadKhosravi/llama3.1-8b-cefr-prefix-tuning-68m-param-matched


cefr_prefix_tuning_68m_best_weights.pt: reconstructing file:   0%|          |  0.00B /  137MB            

cefr_prefix_tuning_68m_best_weights.pt: downloading bytes:           |  0.00B            

✔️ Controller checkpoint downloaded:
/root/.cache/huggingface/hub/models--MohammadKhosravi--llama3.1-8b-cefr-prefix-tuning-68m-param-matched/snapshots/c3bc11c2f3b6f116f113ec94644cbd1ef15c35f5/cefr_prefix_tuning_68m_best_weights.pt
✔️ Strict checkpoint match successful.

[3/4] Loading EFCAMDAT Evaluation Prompt Matrix...
Benchmark conditions: 702


In [ ]:
# =========================================================
# 16. PREFIX-AWARE AUTOREGRESSIVE GENERATION FUNCTION
# =========================================================
def generate_with_cefr_prefix(input_ids, attention_mask, cefr_ids, max_new_tokens=200, temperature=0.6):
    batch_size = input_ids.shape[0]
    padded_prompt_length = input_ids.shape[1]
    num_virtual_tokens = prefix_controller.num_virtual_tokens

    past_key_values = prefix_controller(cefr_ids)

    prefix_mask = torch.ones(batch_size, num_virtual_tokens, dtype=attention_mask.dtype, device=input_ids.device)
    full_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

    position_ids = full_attention_mask.long().cumsum(-1) - 1
    position_ids.masked_fill_(full_attention_mask == 0, 1)
    position_ids = position_ids[:, num_virtual_tokens:]

    cache_position = torch.arange(num_virtual_tokens, num_virtual_tokens + padded_prompt_length, dtype=torch.long, device=input_ids.device)

    outputs = base_model(
        input_ids=input_ids,
        attention_mask=full_attention_mask,
        position_ids=position_ids,
        cache_position=cache_position,
        past_key_values=past_key_values,
        use_cache=True
    )

    next_token_logits = outputs.logits[:, -1, :]
    generated_tokens = []
    finished = torch.zeros(batch_size, dtype=torch.bool, device=input_ids.device)

    next_position_ids = full_attention_mask.long().sum(dim=1, keepdim=True)
    current_cache_length = num_virtual_tokens + padded_prompt_length
    current_attention_mask = full_attention_mask

    for step in range(max_new_tokens):
        logits = next_token_logits / temperature
        probabilities = torch.softmax(logits, dim=-1)
        sampled_tokens = torch.multinomial(probabilities, num_samples=1)

        sampled_tokens = torch.where(finished.unsqueeze(1), torch.full_like(sampled_tokens, tokenizer.pad_token_id), sampled_tokens)
        generated_tokens.append(sampled_tokens)

        newly_finished = (~finished) & (sampled_tokens.squeeze(1) == tokenizer.eos_token_id)
        finished_after_sample = finished | newly_finished

        if finished_after_sample.all():
            finished = finished_after_sample
            break

        active_for_forward = ~finished_after_sample
        input_token = torch.where(active_for_forward.unsqueeze(1), sampled_tokens, torch.full_like(sampled_tokens, tokenizer.pad_token_id))

        new_attention_values = active_for_forward.long().unsqueeze(1)
        current_attention_mask = torch.cat([current_attention_mask, new_attention_values], dim=1)

        forward_position_ids = torch.where(active_for_forward.unsqueeze(1), next_position_ids, torch.ones_like(next_position_ids))
        next_cache_position = torch.tensor([current_cache_length], dtype=torch.long, device=input_ids.device)

        outputs = base_model(
            input_ids=input_token,
            attention_mask=current_attention_mask,
            position_ids=forward_position_ids,
            cache_position=next_cache_position,
            past_key_values=past_key_values,
            use_cache=True
        )

        next_token_logits = outputs.logits[:, -1, :]
        next_position_ids = next_position_ids + active_for_forward.long().unsqueeze(1)
        current_cache_length += 1
        finished = finished_after_sample

    if generated_tokens:
        generated_ids = torch.cat(generated_tokens, dim=1)
    else:
        generated_ids = torch.empty(batch_size, 0, dtype=torch.long, device=input_ids.device)

    return generated_ids

In [ ]:
# =========================================================
# 17. PHASE 1 — GENERATION
# =========================================================
print("\n[4/4] Initiating Batch Generation Phase...")
generated_records = []
base_model.eval()
prefix_controller.eval()

for b_start in tqdm(range(0, len(df), BATCH_SIZE), desc="Processing Prompt Batches"):
    batch_df = df.iloc[b_start : b_start + BATCH_SIZE]
    batch_prompts, batch_cefrs = [], []

    for _, row in batch_df.iterrows():
        target_cefr = str(row["cefr"]).strip().upper()
        batch_cefrs.append(target_cefr)
        batch_prompts.append(build_blind_prompt(row["topic_title"]))

    inputs = tokenizer(batch_prompts, padding=True, return_tensors="pt")
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)
    cefr_tensor = torch.tensor([label_map[level] for level in batch_cefrs], dtype=torch.long, device=device)

    with torch.inference_mode():
        output_ids = generate_with_cefr_prefix(
            input_ids=input_ids,
            attention_mask=attention_mask,
            cefr_ids=cefr_tensor,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE
        )

    for i in range(output_ids.shape[0]):
        gen_text = tokenizer.decode(output_ids[i], skip_special_tokens=True).strip()
        generated_records.append({
            "topic_id": batch_df.iloc[i]["topic_id"],
            "topic_title": batch_df.iloc[i]["topic_title"],
            "target_cefr": batch_cefrs[i],
            "generated_text": gen_text
        })

# =========================================================
# 18. GENERATION SANITY CHECK
# =========================================================
print("\n" + "=" * 70)
print("GENERATION COMPLETE")
print("=" * 70)
print(f"Generated records: {len(generated_records):,}")

for record in generated_records[:3]:
    print(f"\nTarget CEFR: {record['target_cefr']}")
    print(f"Topic: {record['topic_title']}")
    print(f"Generation:\n{record['generated_text']}")
    print("-" * 70)

# =========================================================
# 19. FREE GENERATION MEMORY
# =========================================================
del base_model
del prefix_controller
torch.cuda.empty_cache()
gc.collect()
print("\n✔️ Llama and prefix controller released.")


[4/4] Initiating Batch Generation Phase...


Processing Prompt Batches:   0%|          | 0/22 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



GENERATION COMPLETE
Generated records: 702

Target CEFR: A1
Topic: Taking inventory in the office
Generation:
There are some chairs, desks, computers, keyboards, pens, pencils, a bookshelf, a table, a microwave, a refrigerator, a coffee machine, a TV, a couch, and a bathroom in the office.
----------------------------------------------------------------------

Target CEFR: A2
Topic: Taking inventory in the office
Generation:
I work in the marketing department of an international company. I like my job. I have a very interesting job. I have to work with people from different countries. I have to find new customers. I have to make campaigns to sell our products. I have to talk to people from different countries. I have to speak in English and Spanish. I have to talk to them on the phone and by e-mail.
----------------------------------------------------------------------

Target CEFR: B1
Topic: Taking inventory in the office
Generation:
Hello, I have done the inventory in the office. We

In [ ]:
# =========================================================
# 20. PHASE 2 — EVALUATION
# =========================================================
print("\nInitiating Post-Generation Evaluation Loop...")

nlp = spacy.load("en_core_web_sm")

print("Deploying Custom JointLoss RoBERTa Evaluator...")
JUDGE_MODEL_ID = "MohammadKhosravi/roberta-large-cefr-classifier-JointLoss"
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model = AutoModelForSequenceClassification.from_pretrained(JUDGE_MODEL_ID).to(device)
judge_model.eval()
print("✔️ CEFR evaluator loaded.")

def calculate_mdd(text):
    doc = nlp(text)
    total_dist, tokens = 0, 0
    for token in doc:
        if token.dep_ != "punct":
            total_dist += abs(token.i - token.head.i)
            tokens += 1
    return total_dist / tokens if tokens > 0 else 0.0

def calculate_sentence_drift(text):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10]
    if len(sentences) <= 1: return 0
    inputs = judge_tokenizer(sentences, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.inference_mode():
        preds = torch.argmax(judge_model(**inputs).logits, dim=-1).cpu().numpy()
    return int(np.max(preds) - np.min(preds))

# =========================================================
# 25. EVALUATE GENERATED TEXT
# =========================================================
final_results = []
for record in tqdm(generated_records, desc="Extracting Metrics"):
    gen_text = record["generated_text"] or "Empty response generation failure."
    target_cefr = record["target_cefr"]

    eval_inputs = judge_tokenizer(gen_text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.inference_mode():
        eval_pred = torch.argmax(judge_model(**eval_inputs).logits, dim=-1).item()
    predicted_cefr = inv_label_map[eval_pred]

    is_strict = int(predicted_cefr == target_cefr)
    is_adjacent = int(abs(label_map[predicted_cefr] - label_map[target_cefr]) <= 1)

    try: flesch_score = textstat.flesch_reading_ease(gen_text)
    except Exception: flesch_score = 0.0

    mdd_score = calculate_mdd(gen_text)
    drift_score = calculate_sentence_drift(gen_text)

    final_results.append({
        "topic_id": record["topic_id"],
        "topic_title": record["topic_title"],
        "target_cefr": target_cefr,
        "predicted_cefr": predicted_cefr,
        "strict_match": is_strict,
        "adjacent_match": is_adjacent,
        "mdd": round(mdd_score, 2),
        "readability_flesch": round(flesch_score, 2),
        "sentence_drift_max": drift_score,
        "generated_text": gen_text
    })

# =========================================================
# 26. SAVE BENCHMARK CSV
# =========================================================
df_final = pd.DataFrame(final_results)
df_final.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\n✔️ Benchmark log saved to:\n{OUTPUT_CSV_PATH}")

# =========================================================
# 27. AGGREGATE RESULTS
# =========================================================
y_true = df_final["target_cefr"]
y_pred = df_final["predicted_cefr"]
strict_acc = df_final["strict_match"].mean() * 100
adj_acc = df_final["adjacent_match"].mean() * 100
mae = mean_absolute_error(y_true.map(label_map), y_pred.map(label_map))
avg_mdd = df_final["mdd"].mean()
avg_flesch = df_final["readability_flesch"].mean()
avg_drift = df_final["sentence_drift_max"].mean()

# =========================================================
# 28. REPORT
# =========================================================
report_text = "=" * 70 + "\n"
report_text += " 📊 FINAL COMPILED MACRO-STATISTICS (CEFR PT ~68M PARAM-MATCHED)\n"
report_text += "=" * 70 + "\n"
report_text += f"Total Processed       : {len(df_final)}\n"
report_text += f"Strict Accuracy       : {strict_acc:.2f}%\n"
report_text += f"Adjacent Accuracy     : {adj_acc:.2f}%\n"
report_text += f"Mean Abs Error (MAE)  : {mae:.4f}\n"
report_text += f"Avg MDD Score         : {avg_mdd:.2f}\n"
report_text += f"Avg Reading Ease      : {avg_flesch:.2f}\n"
report_text += f"Avg Sentence Drift    : {avg_drift:.2f} levels\n"
report_text += "Avg Perplexity        : (Computed externally)\n"
report_text += "=" * 70 + "\n\n"

report_text += "📈 PERFORMANCE BREAKDOWN BY CEFR TARGET LEVEL:\n"
level_agg = df_final.groupby("target_cefr").agg(
    strict_accuracy=("strict_match", lambda x: np.mean(x) * 100),
    avg_mdd=("mdd", "mean"),
    avg_flesch=("readability_flesch", "mean"),
    avg_drift=("sentence_drift_max", "mean")
).reindex(["A1", "A2", "B1", "B2", "C1", "C2"]).round(2)
report_text += level_agg.to_string() + "\n\n"

report_text += "=" * 70 + "\n"
report_text += "📝 DETAILED CLASSIFICATION REPORT:\n"
cefr_labels = ["A1", "A2", "B1", "B2", "C1", "C2"]
report_text += classification_report(y_true, y_pred, labels=cefr_labels, zero_division=0)
report_text += "\n" + "=" * 70 + "\n"

with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
    f.write(report_text)
print(report_text)

# =========================================================
# 32. CONFUSION MATRIX
# =========================================================
cm = confusion_matrix(y_true, y_pred, labels=cefr_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=cefr_labels, yticklabels=cefr_labels, cbar=True, square=True)
plt.title("CEFR Alignment (CEFR Multi-Prefix Tuning ~68M Parameter-Matched)", fontsize=12, pad=15)
plt.xlabel("Predicted CEFR Level (JointLoss Evaluator)", fontsize=10, labelpad=10)
plt.ylabel("Target CEFR Level (Dataset Input)", fontsize=10, labelpad=10)
plt.tight_layout()
plt.savefig(OUTPUT_IMG_PATH, dpi=300)
plt.close()
print(f"🎨 Confusion Matrix saved to:\n{OUTPUT_IMG_PATH}")

# =========================================================
# 33. FINAL SUMMARY
# =========================================================
print("\n" + "=" * 70)
print("CEFR 68M PARAMETER-MATCHED EXPERIMENT COMPLETE")
print("=" * 70)
print(f"Strict Accuracy       : {strict_acc:.2f}%")
print(f"Adjacent Accuracy     : {adj_acc:.2f}%")
print(f"MAE                   : {mae:.4f}")
print(f"Average MDD           : {avg_mdd:.2f}")
print(f"Average Flesch        : {avg_flesch:.2f}")
print(f"Average Drift         : {avg_drift:.2f}")
print("-" * 70)
print(f"CSV:\n{OUTPUT_CSV_PATH}")
print(f"\nReport:\n{OUTPUT_TXT_PATH}")
print(f"\nConfusion Matrix:\n{OUTPUT_IMG_PATH}")
print("=" * 70)


Initiating Post-Generation Evaluation Loop...
Deploying Custom JointLoss RoBERTa Evaluator...


config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/388 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✔️ CEFR evaluator loaded.


Extracting Metrics:   0%|          | 0/702 [00:00<?, ?it/s]


✔️ Benchmark log saved to:
/content/drive/MyDrive/Mohammd_Thesis/Results/CEFR_Prefix_Tuning_68M_Param_Matched/cefr_prefix_68m_param_matched_benchmark_results_log.csv
 📊 FINAL COMPILED MACRO-STATISTICS (CEFR PT ~68M PARAM-MATCHED)
Total Processed       : 702
Strict Accuracy       : 45.44%
Adjacent Accuracy     : 68.80%
Mean Abs Error (MAE)  : 1.0940
Avg MDD Score         : 1.78
Avg Reading Ease      : 81.08
Avg Sentence Drift    : 2.38 levels
Avg Perplexity        : (Computed externally)

📈 PERFORMANCE BREAKDOWN BY CEFR TARGET LEVEL:
             strict_accuracy  avg_mdd  avg_flesch  avg_drift
target_cefr                                                 
A1                     62.39     1.46       84.61       1.21
A2                     52.99     1.71       84.34       2.00
B1                     56.41     1.85       80.96       2.48
B2                     50.43     1.82       79.76       2.68
C1                     28.21     1.95       77.52       2.79
C2                     22.22     